# PII NER — training & TFLite export (**Google Colab / GPU** edition)

Fine-tunes `distilbert-base-uncased` for PII token classification and exports the
TFLite model + tokenizer files consumed by the Flutter app.

This is the Colab port of `pii-ner.ipynb`. On a CPU laptop training took ~6 h and
could hang the machine; on a Colab **T4 GPU** the same 3 epochs finish in roughly
**10–20 minutes**. The model logic, hyperparameters, label set and TFLite I/O
contract are **identical** to the local notebook — only the environment setup,
paths, and a final download step changed.

---
## How to run (read this first)

1. **Runtime → Change runtime type → Hardware accelerator = `T4 GPU`** → Save.
2. Run **STEP 1** (the install cell) once.
3. **Runtime → Restart session** when it finishes. *This restart is mandatory* —
   see the note in STEP 1.
4. **Runtime → Run all** (STEP 1 will no-op / re-run harmlessly).
5. At the very end a `pii_ner_assets.zip` downloads to your PC. Unzip it and copy
   the 4 files into the repo's **`assets/`** folder (overwriting the old ones):
   `pii_model_no_quant.tflite`, `vocab.txt`, `tokenizer.json`, `tokenizer_config.json`.

**What ships to the app (must not change):** input
`input_ids[1,128] int32` + `attention_mask[1,128] int32` → output `logits[1,128,35]`.
The final cell asserts this contract before you download.

In [1]:
# =====================================================================
# STEP 1 — Install the EXACT pinned stack (GPU-enabled).
# Run this cell, then RESTART the runtime (Runtime > Restart session)
# BEFORE running anything below.
#
# WHY THE RESTART IS MANDATORY:
#   Colab preloads a newer numpy (2.x) at kernel startup. We downgrade to
#   numpy 1.26 (TF 2.17 requires numpy<2). A downgrade of an already-imported
#   C-extension package only takes effect in a FRESH session. The version-guard
#   cell (STEP 2) will hard-fail if you forget to restart.
#
# WHY THESE PINS (copied verbatim from the local requirements.txt — do NOT bump):
#   * transformers 5.x DROPPED every TF* model class -> must stay on 4.44.x.
#   * transformers' TF models need the Keras-2 API -> tf-keras + TF_USE_LEGACY_KERAS=1
#     (Colab ships Keras 3, which is incompatible; the env var routes around it).
#   * TF 2.17 requires numpy<2 and protobuf<5.
#   * tokenizers / huggingface_hub ranges are what transformers 4.44 needs.
#   * "tensorflow[and-cuda]" pulls TF's OWN matching CUDA/cuDNN wheels, so the
#     GPU is detected regardless of what CUDA Colab happens to ship. This is the
#     reproducible choice — it will not "fight" Colab's system libraries.
# =====================================================================
!pip install -q \
  "tensorflow[and-cuda]==2.17.0" \
  "tf-keras==2.17.0" \
  "transformers==4.44.2" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.24.6" \
  "safetensors==0.4.5" \
  "numpy==1.26.4" \
  "protobuf==4.25.4" \
  "scikit-learn==1.5.2"

print("\n\n" + "=" * 68)
print("Install finished.  NEXT:  Runtime > Restart session,  then  Run all.")
print("=" * 68)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.4/601.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0

In [2]:
!pip uninstall -y jax jaxlib flax

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: flax 0.11.2
Uninstalling flax-0.11.2:
  Successfully uninstalled flax-0.11.2


In [1]:
import os
os.environ["USE_TF"] = "1"      # force transformers onto the TF backend
os.environ["USE_TORCH"] = "0"   # <-- stop transformers importing Colab's broken torch
os.environ["USE_FLAX"] = "0"

In [2]:
# =====================================================================
# STEP 2 — Imports, environment flags, and version/GPU guards.
# TF_USE_LEGACY_KERAS MUST be set BEFORE tensorflow is imported.
# =====================================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # transformers TF models -> Keras 2 (tf-keras)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # hush TF's info/warning spam

import re, glob, json, shutil, pickle
from pathlib import Path
import numpy as np
import tensorflow as tf
import transformers, tokenizers, huggingface_hub
from transformers import AutoTokenizer, TFDistilBertForTokenClassification
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, snapshot_download

# ---- Version guard: fails LOUDLY if the post-install restart was skipped ----
print("tensorflow  :", tf.__version__)
print("numpy       :", np.__version__)
print("transformers:", transformers.__version__)
print("tokenizers  :", tokenizers.__version__)
print("hf_hub      :", huggingface_hub.__version__)
assert tf.__version__.startswith("2.17"),     "TF is not 2.17 — re-run STEP 1 then RESTART."
assert np.__version__.startswith("1.26"),     "numpy is not 1.26 — you skipped the RESTART after STEP 1."
assert transformers.__version__ == "4.44.2",  "transformers is not 4.44.2 — re-run STEP 1 then RESTART."

# ---- GPU guard: this notebook is pointless on CPU (6 h vs ~15 min) ----------
gpus = tf.config.list_physical_devices("GPU")
print("GPUs        :", gpus)
assert gpus, "No GPU visible! Runtime > Change runtime type > T4 GPU, then Run all."

tf.random.set_seed(42)
np.random.seed(42)
print("\nEnvironment OK — GPU ready.")

tensorflow  : 2.17.0
numpy       : 1.26.4
transformers: 4.44.2
tokenizers  : 0.19.1
hf_hub      : 0.24.6
GPUs        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Environment OK — GPU ready.


In [3]:
# =====================================================================
# STEP 3 — Paths.
# On Colab there is no git repo, so we work under /content and collect the
# 4 shippable files into ASSETS_DIR, which gets zipped & downloaded at the end.
#
# OPTIONAL persistence: set USE_DRIVE = True to write outputs to your Google
# Drive instead of ephemeral /content. Handy if the runtime disconnects — the
# trained checkpoint survives so you can re-run only the TFLite export. Default
# is False (simplest; everything is downloaded as a zip at the end anyway).
# =====================================================================
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/pii_ner")
else:
    BASE_DIR = Path("/content/pii_ner")

DATA_DIR   = BASE_DIR / "data"       # dataset cache
MODEL_DIR  = BASE_DIR / "pii_model"  # HF checkpoint + tokenizer
ASSETS_DIR = BASE_DIR / "assets"     # <-- the 4 files you copy into the app's assets/
for d in (DATA_DIR, MODEL_DIR, ASSETS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR  :", BASE_DIR)
print("DATA_DIR  :", DATA_DIR)
print("MODEL_DIR :", MODEL_DIR)
print("ASSETS_DIR:", ASSETS_DIR)

BASE_DIR  : /content/pii_ner
DATA_DIR  : /content/pii_ner/data
MODEL_DIR : /content/pii_ner/pii_model
ASSETS_DIR: /content/pii_ner/assets


In [4]:
# ---- Configuration (env-overridable; same defaults as the tested local run) --
# NOTE: on GPU a T4 can comfortably handle BATCH_SIZE=32 (~2x faster). We keep 16
# to reproduce the exact hyperparameters you already validated locally; bump it
# only if you want speed and are ok with a slightly different optimisation path.
SAMPLE_SIZE    = int(os.environ.get("PII_SAMPLE_SIZE", 30000))  # examples used
MAX_LENGTH     = 128            # MUST stay 128 to match the Flutter app I/O
BATCH_SIZE     = 16
EPOCHS         = int(os.environ.get("PII_EPOCHS", 3))
LEARNING_RATE  = 2e-5
VAL_SPLIT      = 0.2
COMPARE_SUBSET = int(os.environ.get("PII_COMPARE_SUBSET", 400))  # #val examples for TFLite compare
print(f"SAMPLE_SIZE={SAMPLE_SIZE}  EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}  "
      f"MAX_LENGTH={MAX_LENGTH}  COMPARE_SUBSET={COMPARE_SUBSET}")

SAMPLE_SIZE=30000  EPOCHS=3  BATCH_SIZE=16  MAX_LENGTH=128  COMPARE_SUBSET=400


In [5]:
# ---- Label set: EXACT same names AND order as the Flutter app (35 labels) --
# Do NOT reorder — the app maps output indices -> these labels positionally.
LABEL_NAMES = ['O', 'B-SURNAME', 'I-SURNAME', 'B-CITY', 'I-CITY', 'B-GIVENNAME', 'I-GIVENNAME',
               'B-DATEOFBIRTH', 'I-DATEOFBIRTH', 'B-DRIVERLICENSENUM', 'I-DRIVERLICENSENUM',
               'B-CREDITCARDNUMBER', 'I-CREDITCARDNUMBER', 'B-TELEPHONENUM', 'I-TELEPHONENUM',
               'B-SOCIALNUM', 'I-SOCIALNUM', 'B-ZIPCODE', 'I-ZIPCODE', 'B-IDCARDNUM', 'I-IDCARDNUM',
               'B-USERNAME', 'I-USERNAME', 'B-STREET', 'I-STREET', 'B-TAXNUM', 'I-TAXNUM',
               'B-EMAIL', 'I-EMAIL', 'B-BUILDINGNUM', 'I-BUILDINGNUM', 'B-PASSWORD', 'I-PASSWORD',
               'B-ACCOUNTNUM', 'I-ACCOUNTNUM']
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_NAMES)}
ID_TO_LABEL = {i: l for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS  = len(LABEL_NAMES)
ENTITY_TYPES = {l[2:] for l in LABEL_NAMES if l != 'O'}
assert NUM_LABELS == 35
print(NUM_LABELS, "labels;", len(ENTITY_TYPES), "entity types")

35 labels; 17 entity types


In [6]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def download_english_train():
    """Download ONLY the English train shard(s) into DATA_DIR (keeps it small).
    Returns a list of local .jsonl file paths. Dataset is public — no HF login."""
    api = HfApi()
    files = api.list_repo_files("ai4privacy/pii-masking-400k", repo_type="dataset")
    jsonl = [f for f in files if f.endswith('.jsonl')]
    def is_en(f):
        bn = os.path.basename(f).lower()
        return bn.endswith('en.jsonl') or 'english' in bn   # e.g. '1en.jsonl'
    en_train = [f for f in jsonl if 'train' in f.lower() and is_en(f)]
    if not en_train:            # fallbacks if naming differs
        en_train = [f for f in jsonl if is_en(f)] or [f for f in jsonl if 'train' in f.lower()] or jsonl
    print("Fetching:", en_train)
    snap = snapshot_download("ai4privacy/pii-masking-400k", repo_type="dataset",
                             cache_dir=str(DATA_DIR), allow_patterns=en_train)
    return sorted(glob.glob(os.path.join(snap, "**", "*.jsonl"), recursive=True))

def load_records(paths, limit=None):
    """Load jsonl records (each has source_text + privacy_mask)."""
    records = []
    for fp in paths:
        with open(fp, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                records.append(json.loads(line))
                if limit and len(records) >= limit:
                    return records
    return records

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
def encode_example(record):
    """Tokenize source_text ONCE and align BIO labels via char offsets.

    Special tokens ([CLS]/[SEP]) and padding get label -100 so they are ignored
    by the masked loss. Returns (input_ids, attention_mask, label_ids), each
    length MAX_LENGTH."""
    text = record.get("source_text", "") or ""
    spans = record.get("privacy_mask", []) or []

    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding="max_length",
                    return_offsets_mapping=True, return_special_tokens_mask=True)
    offsets = enc["offset_mapping"]
    special = enc["special_tokens_mask"]

    labels = []
    for (s, e), sp in zip(offsets, special):
        # special token or zero-width (padding) -> ignore in loss
        labels.append(-100 if (sp == 1 or s == e) else LABEL_TO_ID['O'])

    for ent in spans:
        etype = str(ent.get("label", "")).upper()
        if etype not in ENTITY_TYPES:
            continue
        es, ee = ent["start"], ent["end"]
        first = True
        for i, (s, e) in enumerate(offsets):
            if labels[i] == -100:
                continue
            if s < ee and e > es:                       # token overlaps entity span
                labels[i] = LABEL_TO_ID[f"B-{etype}" if first else f"I-{etype}"]
                first = False
    return enc["input_ids"], enc["attention_mask"], labels

def build_arrays(records):
    ii, am, lb = [], [], []
    for r in records:
        a, b, c = encode_example(r)
        ii.append(a); am.append(b); lb.append(c)
    return (np.asarray(ii, dtype=np.int32),
            np.asarray(am, dtype=np.int32),
            np.asarray(lb, dtype=np.int32))

In [8]:
# ---- Entity-level metrics (span match), no external deps -------------------
def extract_spans(tags):
    """BIO tag list -> set of (type, start, end) spans."""
    spans, cur = [], None
    for i, tag in enumerate(tags):
        if tag == 'O':
            if cur: spans.append(tuple(cur)); cur = None
            continue
        prefix, _, etype = tag.partition('-')
        if prefix == 'B':
            if cur: spans.append(tuple(cur))
            cur = [etype, i, i]
        else:  # 'I' — extend if same type, else treat leniently as a new span
            if cur and cur[0] == etype:
                cur[2] = i
            else:
                if cur: spans.append(tuple(cur))
                cur = [etype, i, i]
    if cur: spans.append(tuple(cur))
    return set(spans)

def entity_f1(true_id_rows, pred_id_rows):
    """Micro P/R/F1 over entity spans; positions with true==-100 are ignored."""
    tp = fp = fn = 0
    per = {}
    for t_row, p_row in zip(true_id_rows, pred_id_rows):
        t_tags, p_tags = [], []
        for t, p in zip(t_row, p_row):
            if int(t) == -100:
                continue
            t_tags.append(ID_TO_LABEL[int(t)])
            p_tags.append(ID_TO_LABEL[int(p)])
        ts, ps = extract_spans(t_tags), extract_spans(p_tags)
        tp += len(ts & ps); fp += len(ps - ts); fn += len(ts - ps)
        for typ, a, b in ts:
            d = per.setdefault(typ, [0, 0, 0]); d[2] += 1
        for span in (ts & ps):
            per[span[0]][0] += 1
    def prf(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        return p, r, f
    p, r, f = prf(tp, fp, fn)
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}

def print_entity_report(true_id_rows, pred_id_rows):
    m = entity_f1(true_id_rows, pred_id_rows)
    print(f"Entity micro  P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}  "
          f"(tp={m['tp']} fp={m['fp']} fn={m['fn']})")
    return m

In [9]:
paths = download_english_train()
records = load_records(paths, limit=SAMPLE_SIZE)
print(f"Loaded {len(records)} records")
print("Sample keys:", list(records[0].keys()) if records else "none")

# sanity: show label alignment on one example
if records:
    ii0, am0, lb0 = encode_example(records[0])
    toks = tokenizer.convert_ids_to_tokens(ii0)
    shown = [(t, ID_TO_LABEL[l]) for t, l in zip(toks, lb0) if l != -100][:25]
    print("source_text:", records[0]['source_text'][:120])
    print("aligned (token,label):", shown)

Fetching: ['data/train/1en.jsonl']


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

1en.jsonl:   0%|          | 0.00/84.8M [00:00<?, ?B/s]

Loaded 30000 records
Sample keys: ['source_text', 'locale', 'language', 'split', 'privacy_mask', 'uid', 'masked_text', 'mbert_tokens', 'mbert_token_classes']
source_text: <p>My child faozzsd379223 (DOB: May/58) will undergo treatment with Dr. faozzsd379223, office at Hill Road. Our ZIP code
aligned (token,label): [('<', 'O'), ('p', 'O'), ('>', 'O'), ('my', 'O'), ('child', 'O'), ('fa', 'B-USERNAME'), ('##oz', 'I-USERNAME'), ('##z', 'I-USERNAME'), ('##sd', 'I-USERNAME'), ('##37', 'I-USERNAME'), ('##9', 'I-USERNAME'), ('##22', 'I-USERNAME'), ('##3', 'I-USERNAME'), ('(', 'O'), ('do', 'O'), ('##b', 'O'), (':', 'O'), ('may', 'B-DATEOFBIRTH'), ('/', 'I-DATEOFBIRTH'), ('58', 'I-DATEOFBIRTH'), (')', 'O'), ('will', 'O'), ('undergo', 'O'), ('treatment', 'O'), ('with', 'O')]


In [10]:
input_ids, attention_masks, label_ids = build_arrays(records)
print("input_ids:", input_ids.shape, "attention_masks:", attention_masks.shape, "labels:", label_ids.shape)
# label distribution sanity (excluding ignored -100)
uniq, cnt = np.unique(label_ids[label_ids != -100], return_counts=True)
nonO = {ID_TO_LABEL[int(u)]: int(c) for u, c in zip(uniq, cnt) if int(u) != 0}
print("non-O label counts:", dict(sorted(nonO.items(), key=lambda x: -x[1])))

input_ids: (30000, 128) attention_masks: (30000, 128) labels: (30000, 128)
non-O label counts: {'I-EMAIL': 43547, 'I-TELEPHONENUM': 29857, 'I-USERNAME': 26561, 'I-ACCOUNTNUM': 21514, 'I-IDCARDNUM': 17371, 'I-GIVENNAME': 13870, 'I-SURNAME': 13160, 'I-SOCIALNUM': 12622, 'I-DATEOFBIRTH': 12147, 'I-PASSWORD': 12124, 'I-DRIVERLICENSENUM': 11763, 'I-CREDITCARDNUMBER': 10411, 'I-TAXNUM': 9619, 'I-CITY': 9499, 'B-GIVENNAME': 9129, 'I-ZIPCODE': 8357, 'B-SURNAME': 6901, 'I-STREET': 6178, 'B-CITY': 6049, 'B-USERNAME': 5406, 'B-EMAIL': 5045, 'B-TELEPHONENUM': 3983, 'B-BUILDINGNUM': 3255, 'B-IDCARDNUM': 3174, 'B-ZIPCODE': 2950, 'B-DATEOFBIRTH': 2942, 'B-STREET': 2864, 'B-ACCOUNTNUM': 2728, 'B-SOCIALNUM': 2293, 'I-BUILDINGNUM': 2084, 'B-PASSWORD': 2031, 'B-TAXNUM': 1789, 'B-DRIVERLICENSENUM': 1601, 'B-CREDITCARDNUMBER': 1215}


In [11]:
train_ii, val_ii, train_am, val_am, train_lb, val_lb = train_test_split(
    input_ids, attention_masks, label_ids, test_size=VAL_SPLIT, random_state=42)
print("train:", train_ii.shape[0], " val:", val_ii.shape[0])

train: 24000  val: 6000


In [12]:
# ---- Masked loss/metric: ignore -100 (padding + special tokens) -----------
def masked_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, -100)
    y_safe = tf.where(mask, y_true, tf.zeros_like(y_true))
    per_tok = tf.keras.losses.sparse_categorical_crossentropy(y_safe, y_pred, from_logits=True)
    mask_f = tf.cast(mask, per_tok.dtype)
    return tf.reduce_sum(per_tok * mask_f) / tf.maximum(tf.reduce_sum(mask_f), 1.0)

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, -100)
    preds = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    hits = tf.logical_and(mask, tf.equal(preds, tf.where(mask, y_true, tf.zeros_like(y_true))))
    return tf.reduce_sum(tf.cast(hits, tf.float32)) / tf.maximum(tf.reduce_sum(tf.cast(mask, tf.float32)), 1.0)

def build_model():
    backbone = TFDistilBertForTokenClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=NUM_LABELS,
        id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)
    ids  = tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int32, name='input_ids')
    mask = tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int32, name='attention_mask')
    logits = backbone(input_ids=ids, attention_mask=mask).logits
    model = tf.keras.Model([ids, mask], logits)
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss=masked_loss, metrics=[masked_accuracy])
    return model, backbone

class EntityF1Callback(tf.keras.callbacks.Callback):
    """Computes val entity-F1 each epoch -> logs['val_f1'] (for EarlyStopping)."""
    def __init__(self, vi, vm, vl):
        super().__init__(); self.vi, self.vm, self.vl = vi, vm, vl
    def on_epoch_end(self, epoch, logs=None):
        logits = self.model.predict({'input_ids': self.vi, 'attention_mask': self.vm},
                                    batch_size=BATCH_SIZE, verbose=0)
        preds = np.asarray(logits).argmax(-1)
        m = entity_f1(self.vl, preds)
        logs = logs or {}
        logs['val_f1'] = m['f1']
        print(f"  val entity-F1={m['f1']:.4f} (P={m['precision']:.3f} R={m['recall']:.3f})")

In [13]:
# ---- Train. On a T4 GPU the "ETA ~2h" you saw on CPU collapses to minutes. ----
# The first PyTorch->TF weight-load prints a few "weights not used / newly
# initialized" messages — that is EXPECTED (we're adding a fresh classifier head).
model, backbone = build_model()

callbacks = [
    EntityF1Callback(val_ii, val_am, val_lb),
    tf.keras.callbacks.EarlyStopping(monitor='val_f1', mode='max', patience=2,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=1, factor=0.5),
]

history = model.fit(
    {'input_ids': train_ii, 'attention_mask': train_am}, train_lb,
    validation_data=({'input_ids': val_ii, 'attention_mask': val_am}, val_lb),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForTokenClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForTokenClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able t

Epoch 1/3
1500/1500 [==============================] - 391s 248ms/step - loss: 0.2762 - masked_accuracy: 0.9277 - val_loss: 0.0944 - val_masked_accuracy: 0.9665 - val_f1: 0.8061 - lr: 2.0000e-05
Epoch 2/3
1500/1500 [==============================] - 370s 247ms/step - loss: 0.0822 - masked_accuracy: 0.9720 - val_loss: 0.0730 - val_masked_accuracy: 0.9736 - val_f1: 0.8479 - lr: 2.0000e-05
Epoch 3/3
1500/1500 [==============================] - 371s 247ms/step - loss: 0.0564 - masked_accuracy: 0.9803 - val_loss: 0.0681 - val_masked_accuracy: 0.9764 - val_f1: 0.8652 - lr: 2.0000e-05


In [14]:
# Final validation entity-level report
val_logits = model.predict({'input_ids': val_ii, 'attention_mask': val_am},
                           batch_size=BATCH_SIZE, verbose=0)
val_pred = np.asarray(val_logits).argmax(-1)
print("Validation:")
_ = print_entity_report(val_lb, val_pred)

Validation:
Entity micro  P=0.8490  R=0.8819  F1=0.8652  (tp=11119 fp=1977 fn=1489)


In [15]:
def predict_sentence(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length',
                    return_tensors='np')
    logits = model.predict({'input_ids': enc['input_ids'].astype(np.int32),
                            'attention_mask': enc['attention_mask'].astype(np.int32)}, verbose=0)
    ids = np.asarray(logits)[0].argmax(-1)
    toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    ents, cur = [], None
    for tok, lid in zip(toks, ids):
        if tok in ('[CLS]', '[SEP]', '[PAD]'):
            continue
        lab = ID_TO_LABEL[int(lid)]
        if lab.startswith('B-'):
            if cur: ents.append(cur)
            cur = {'text': tok.replace('##', ''), 'label': lab[2:]}
        elif lab.startswith('I-') and cur:
            cur['text'] += tok.replace('##', '')
        else:
            if cur: ents.append(cur); cur = None
    if cur: ents.append(cur)
    return ents

for s in ["My name is John Smith and my email is john.smith@email.com",
          "Please call me at 555-123-4567 or visit me at 123 Main Street",
          "My credit card number is 4532-1234-5678-9012"]:
    print(s)
    for e in predict_sentence(s):
        print("   -", e['text'], "(", e['label'], ")")

My name is John Smith and my email is john.smith@email.com
   - john ( GIVENNAME )
   - smith ( SURNAME )
   - john.smith@email.com ( EMAIL )
Please call me at 555-123-4567 or visit me at 123 Main Street
   - 555-123-4567 ( TELEPHONENUM )
   - 123 ( BUILDINGNUM )
   - mainstreet ( STREET )
My credit card number is 4532-1234-5678-9012
   - 4532-1234-5678-9012 ( TELEPHONENUM )


In [16]:
# Save HF checkpoint + tokenizer + label mappings into MODEL_DIR.
backbone.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
with open(MODEL_DIR / 'label_mappings.pkl', 'wb') as f:
    pickle.dump({'label_to_id': LABEL_TO_ID, 'id_to_label': ID_TO_LABEL, 'label_names': LABEL_NAMES}, f)
print("Saved checkpoint to", MODEL_DIR)

Saved checkpoint to /content/pii_ner/pii_model


## TFLite export — compress, compare, ship the winner

Convert the trained model to **two** candidates (no-quant float32 and
dynamic-range int8 weights), score each on a held-out subset with entity-F1,
compare against size, then copy **only the winner** to `ASSETS_DIR` under the
filename the app loads (`pii_model_no_quant.tflite`). I/O signature stays
`input_ids[1,128] int32 + attention_mask[1,128] int32 -> logits[1,128,35]`.

In [17]:
def _concrete_func(model):
    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name='input_ids'),
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name='attention_mask')])
    def f(input_ids, attention_mask):
        return model(input_ids=input_ids, attention_mask=attention_mask, training=False).logits
    return f.get_concrete_function()

def convert_tflite(hf_model, dynamic_range: bool, out_path: Path):
    conv = tf.lite.TFLiteConverter.from_concrete_functions([_concrete_func(hf_model)],
                                                           trackable_obj=hf_model)
    if dynamic_range:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]   # int8 weights only
    tfl = conv.convert()
    out_path.write_bytes(tfl)
    return len(tfl) / 1024 / 1024  # MB

def eval_tflite_f1(tflite_path, vi, vm, vl, n):
    interp = tf.lite.Interpreter(model_path=str(tflite_path)); interp.allocate_tensors()
    inp = interp.get_input_details(); out = interp.get_output_details()
    idx = {'input_ids': None, 'attention_mask': None}
    for d in inp:
        for k in idx:
            if k in d['name']:
                idx[k] = d['index']
    if idx['input_ids'] is None:   # fallback by order
        idx['input_ids'], idx['attention_mask'] = inp[1]['index'], inp[0]['index']
    preds = []
    for i in range(min(n, vi.shape[0])):
        interp.set_tensor(idx['input_ids'], vi[i:i+1].astype(np.int32))
        interp.set_tensor(idx['attention_mask'], vm[i:i+1].astype(np.int32))
        interp.invoke()
        preds.append(interp.get_tensor(out[0]['index'])[0].argmax(-1))
    preds = np.asarray(preds)
    return entity_f1(vl[:preds.shape[0]], preds), out[0]['shape']

In [18]:
hf = TFDistilBertForTokenClassification.from_pretrained(str(MODEL_DIR))

cand_dir = BASE_DIR / "_tflite_candidates"; cand_dir.mkdir(exist_ok=True)
no_quant_p = cand_dir / "pii_model_no_quant.tflite"
dynamic_p  = cand_dir / "pii_model_dynamic.tflite"

print("Converting candidates...")
size_nq = convert_tflite(hf, dynamic_range=False, out_path=no_quant_p)
size_dy = convert_tflite(hf, dynamic_range=True,  out_path=dynamic_p)

print(f"Scoring on {COMPARE_SUBSET} val examples...")
f1_nq, shape_nq = eval_tflite_f1(no_quant_p, val_ii, val_am, val_lb, COMPARE_SUBSET)
f1_dy, shape_dy = eval_tflite_f1(dynamic_p,  val_ii, val_am, val_lb, COMPARE_SUBSET)

print("\n%-14s %8s %10s  %s" % ("candidate", "size(MB)", "entityF1", "out_shape"))
print("%-14s %8.1f %10.4f  %s" % ("no_quant", size_nq, f1_nq['f1'], list(shape_nq)))
print("%-14s %8.1f %10.4f  %s" % ("dynamic",  size_dy, f1_dy['f1'], list(shape_dy)))

# Decision: prefer higher F1; if within 0.5 F1-point, prefer the smaller model.
if f1_dy['f1'] >= f1_nq['f1'] - 0.005:
    winner, wsize, wf1 = dynamic_p, size_dy, f1_dy['f1']
    print(f"\nWINNER: dynamic-range ({wsize:.1f} MB, F1={wf1:.4f}) — near-equal F1, much smaller")
else:
    winner, wsize, wf1 = no_quant_p, size_nq, f1_nq['f1']
    print(f"\nWINNER: no-quant ({wsize:.1f} MB, F1={wf1:.4f}) — higher F1")
print("(review the numbers above; you have final say on which ships)")

Some layers from the model checkpoint at /content/pii_ner/pii_model were not used when initializing TFDistilBertForTokenClassification: ['dropout_19']
- This IS expected if you are initializing TFDistilBertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForTokenClassification were not initialized from the model checkpoint at /content/pii_ner/pii_model and are newly initialized: ['dropout_39']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Converting candidates...


Scoring on 400 val examples...

candidate      size(MB)   entityF1  out_shape
no_quant          252.2     0.8477  [1, 128, 35]
dynamic            64.0     0.8418  [1, 128, 35]

WINNER: no-quant (252.2 MB, F1=0.8477) — higher F1
(review the numbers above; you have final say on which ships)


In [22]:
winner, wsize, wf1 = dynamic_p, size_dy, f1_dy['f1']
# Ship the winner under the filename the app loads; sync tokenizer files too.
target = ASSETS_DIR / "pii_model.tflite"
shutil.copyfile(winner, target)
for fn in ("vocab.txt", "tokenizer.json", "tokenizer_config.json"):
    src = MODEL_DIR / fn
    if src.exists():
        shutil.copyfile(src, ASSETS_DIR / fn)
print("Exported:", target, f"({target.stat().st_size/1024/1024:.1f} MB)")
print("Collected files in", ASSETS_DIR, ":", [p.name for p in ASSETS_DIR.iterdir()])

Exported: /content/pii_ner/assets/pii_model.tflite (64.0 MB)
Collected files in /content/pii_ner/assets : ['tokenizer_config.json', 'vocab.txt', 'pii_model.tflite', 'tokenizer.json', 'pii_model_no_quant.tflite']


In [23]:
# Verify the shipped model matches the app contract: [1,128,35] logits.
interp = tf.lite.Interpreter(model_path=str(ASSETS_DIR / 'pii_model.tflite'))
interp.allocate_tensors()
outd = interp.get_output_details()[0]
print("Output shape:", list(outd['shape']), "(expected [1, 128, 35])")
assert list(outd['shape']) == [1, MAX_LENGTH, NUM_LABELS], "Output signature changed — app would break!"
print("OK — model I/O matches the Flutter app contract.")

Output shape: [1, 128, 35] (expected [1, 128, 35])
OK — model I/O matches the Flutter app contract.


## Download to your PC

The next cell zips the 4 shippable files and downloads `pii_ner_assets.zip`.
Unzip it and copy the files into the repo's **`assets/`** folder, overwriting the
existing ones:

- `pii_model_no_quant.tflite`
- `vocab.txt`
- `tokenizer.json`
- `tokenizer_config.json`

(If the browser blocks the auto-download, open the Files panel on the left,
find `/content/pii_ner_assets.zip`, and download it manually.)

In [24]:
# Zip the assets and trigger a browser download.
zip_base = "/content/pii_ner_assets"
if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")
shutil.make_archive(zip_base, 'zip', ASSETS_DIR)
print("Zipped:", zip_base + ".zip",
      f"({os.path.getsize(zip_base + '.zip')/1024/1024:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("Auto-download failed — grab it from the Files panel at",
          zip_base + ".zip", "\nReason:", e)

Zipped: /content/pii_ner_assets.zip (101.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>